# E04 Breast Cancer（Wisconsin）：量纲、PCA 与召回权衡

> Issue #9 | [UCI Breast Cancer Wisconsin](https://archive.ics.uci.edu/ml/datasets/breast+cancer+wisconsin+%28diagnostic%29)

## 你将学到
1. **医学二分类**：恶性漏诊（假阴性 FN）代价远高于误报（FP）→ 优先看 **Recall**
2. **量纲与缩放**：radius/area 等量纲差异大；距离/线性模型前必须 `StandardScaler`
3. **PCA 理论**：用正交主轴压缩方差；`mean_*` / `worst_*` / `error_*` 高度共线（多重共线性）
4. **`FeatureImportanceAnalyzer`**：`method='builtin'` 读取树模型分裂增益
5. **分层 CV + 混淆矩阵**：FN 即「将恶性判为良性」

## 指标
- Precision–Recall 权衡；阈值非默认 0.5
- `prediction_type='label'` 提交练习


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 加载数据


In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
print(df.shape)
print('target counts:\n', df['target'].value_counts())
print('classes:', list(data.target_names), '— 通常 0=malignant, 1=benign（以 sklearn 为准）')
df.head(2)


## 2. EDA：量纲跨度与缺失


In [ ]:
from wanglaoshi import DataAnalyzer

analyzer = DataAnalyzer(df)
display(analyzer.basic_statistics().head(10))
display(analyzer.missing_value_analysis())
feat = df.drop(columns=['target'])
print('mean feature scale ratio max/min:', feat.mean().max() / max(feat.mean().min(), 1e-9))


### 2.1 多重共线性：mean / worst / error 特征族

同一几何量（如 radius）的均值、标准误、极值往往高度相关，线性模型系数不稳定 → PCA 或正则化。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cols_sample = [c for c in feat.columns if 'radius' in c or 'area' in c][:6]
if len(cols_sample) >= 2:
    plt.figure(figsize=(5, 4))
    sns.heatmap(feat[cols_sample].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
    plt.title('Multicollinearity among radius/area family')
    plt.tight_layout()
    plt.show()


## 3. 分层划分


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

X = feat
y = df['target']
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print('train counts', ytr.value_counts().to_dict())
print('test counts ', yte.value_counts().to_dict())


## 4. Pipeline：Scaler → PCA → LogisticRegression


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

n_comp = 10 if QUICK else 15
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=n_comp, random_state=RANDOM_STATE)),
    ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
pipe.fit(Xtr, ytr)
proba = pipe.predict_proba(Xte)[:, 1]
pred = (proba >= 0.5).astype(int)
print(classification_report(yte, pred, target_names=list(data.target_names)))
print('ROC-AUC', roc_auc_score(yte, proba))
print('AP', average_precision_score(yte, proba))
print('PCA n_components=', n_comp)


### 4.1 PCA 方差解释（在训练集 fit）


In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(StandardScaler().fit_transform(Xtr))
cum = np.cumsum(pca_full.explained_variance_ratio_)
k95 = int(np.searchsorted(cum, 0.95) + 1)
print(f'components for 95% variance: {k95} / {Xtr.shape[1]}')


## 5. 阈值扫描：提高 Recall（少漏诊）


In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score

rows = []
for thr in [0.15, 0.25, 0.35, 0.5, 0.65]:
    p = (proba >= thr).astype(int)
    rows.append({
        'thr': thr,
        'precision': precision_score(yte, p, zero_division=0),
        'recall': recall_score(yte, p, zero_division=0),
        'f1': f1_score(yte, p, zero_division=0),
    })
thr_df = pd.DataFrame(rows)
display(thr_df)

from sklearn.metrics import precision_recall_curve
prec, rec, _ = precision_recall_curve(yte, proba)
plt.figure(figsize=(5, 4))
plt.plot(rec, prec)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('PR curve — move along curve = change threshold')
plt.grid(True, alpha=0.3)
plt.show()


## 6. 混淆矩阵：假阴性 FN 的代价


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(yte, pred)
print('confusion matrix [[TN, FP], [FN, TP]] assuming 0=malignant, 1=benign:')
print(cm)
print('FN = 恶性被判良性 —— 医学场景最危险')


## 7. QuickBaseline（缩放后全特征）


In [ ]:
from wanglaoshi import CompetitionTools as CT

scaler = StandardScaler()
Xtr_s = pd.DataFrame(scaler.fit_transform(Xtr), columns=Xtr.columns, index=Xtr.index)
Xte_s = pd.DataFrame(scaler.transform(Xte), columns=Xte.columns, index=Xte.index)
baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr_s, ytr, Xte_s, yte, task_type='classification')
comparison = baseline.compare_models(results, top_n=6)
comparison


## 8. FeatureImportanceAnalyzer（builtin，树模型）


In [ ]:
from sklearn.ensemble import RandomForestClassifier

best = comparison.iloc[0]['Model']
model = results[best]['model']
# 若最优非树模型，换 RF 演示 builtin
if not hasattr(model, 'feature_importances_'):
    model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
    model.fit(Xtr_s, ytr)

fia = CT.FeatureImportanceAnalyzer()
imp = fia.calculate_importance(model, Xtr_s, ytr, method='builtin')
display(imp.head(12))


### 8.1 分层交叉验证稳定性


In [ ]:
from sklearn.ensemble import RandomForestClassifier as RFC

clf_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RFC(n_estimators=50 if QUICK else 100, random_state=RANDOM_STATE)),
])
cv = StratifiedKFold(n_splits=3 if QUICK else 5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(clf_cv, X, y, cv=cv, scoring='roc_auc')
print(f'stratified CV ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}')


### 8.2 全特征相关矩阵抽样（共线强度）


In [ ]:
sample_feats = list(feat.columns[:12])
plt.figure(figsize=(6, 5))
sns.heatmap(feat[sample_feats].corr(), cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Feature block correlation (first 12 cols)')
plt.tight_layout()
plt.show()


### 8.3 无 PCA vs 有 PCA：Quick 对比


In [ ]:
from sklearn.metrics import accuracy_score

pipe_full = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
pipe_full.fit(Xtr, ytr)
acc_full = accuracy_score(yte, pipe_full.predict(Xte))
acc_pca = accuracy_score(yte, pred)
print('test accuracy no-PCA logistic:', acc_full)
print('test accuracy PCA logistic:  ', acc_pca)
print('PCA 降维可能略损精度但加速推理、缓解共线')


### 8.4 Permutation importance（子集，QUICK）


In [ ]:
_sub = Xtr_s.sample(min(80, len(Xtr_s)), random_state=RANDOM_STATE)
imp_perm = fia.calculate_importance(
    model, _sub, ytr.loc[_sub.index],
    method='permutation', n_repeats=3 if QUICK else 5,
)
display(imp_perm.head(10))


## 9. 模拟提交


In [ ]:
from pathlib import Path

pred_best = np.asarray(results[best]['predictions'])
out = Path('outputs/04_breast_cancer')
out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred_best)), 'target': yte.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred_best, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))
sub.head()


### 9.1 敏感性：降低阈值对 FN 的影响


In [ ]:
thr_low = 0.25
pred_low = (proba >= thr_low).astype(int)
cm_default = confusion_matrix(yte, pred)
cm_low = confusion_matrix(yte, pred_low)
# sklearn: target 0=malignant, 1=benign —— FN = 恶性被判良性
fn_default = int(cm_default[0, 1])
fn_low = int(cm_low[0, 1])
print('FN count default thr 0.5:', fn_default)
print('FN count low thr 0.25:', fn_low)
print('扩大 Recall 通常减少 FN（恶性漏检）但增加 FP')


### 9.2 worst-radius 家族特征散点


In [ ]:
wcol = 'worst radius' if 'worst radius' in feat.columns else feat.columns[0]
plt.figure(figsize=(5, 3.5))
for t in sorted(df['target'].unique()):
    plt.hist(df.loc[df['target'] == t, wcol], bins=20, alpha=0.5, label=data.target_names[int(t)])
plt.xlabel(wcol)
plt.ylabel('count')
plt.title('Feature distribution by diagnosis')
plt.legend()
plt.tight_layout()
plt.show()


## 10. 检查清单
- [x] 医学 Recall 与阈值
- [x] 缩放 + PCA + 共线性讨论
- [x] FeatureImportanceAnalyzer builtin
- [x] 分层 CV + FN 代价
- [x] SubmissionGenerator
- [ ] 概率校准后再定阈值


## 附录 A：医学分类的代价非对称

假阴性（漏诊恶性）代价通常高于假阳性（额外检查）。因此除 Accuracy 外，必须看 **Recall（敏感性）** 与 PR 曲线思维。

Wisconsin Breast Cancer（sklearn）特征含 mean / se / worst 三组，彼此高度相关 → PCA 或特征选择有意义。


In [ ]:
from sklearn.metrics import recall_score, precision_score, confusion_matrix
# 若前面已有 y_valid / pred
try:
    print('recall(malignant class depends on encoding):', recall_score(y_valid, pred, pos_label=0 if set(np.unique(y_valid))<={0,1} else 1, zero_division=0))
    print('confusion:\n', confusion_matrix(y_valid, pred))
except Exception as e:
    print('metric appendix uses existing names if available:', e)
    try:
        print(confusion_matrix(yte, pred))
    except Exception:
        pass


## 附录 B：多重共线性与可解释性

相关很高时，单系数符号不稳定；树模型的 impurity importance 也会被相关特征分摊。排列重要性（`FeatureImportanceAnalyzer` method=`permutation`）更可靠但更慢。


## 附录 C：从「刷分」到「可部署指标」

报告模板（作业可抄）：
1. 验证协议：分层 holdout / StratifiedKFold，种子固定
2. 主指标：Recall@固定 Precision 或 F1；辅指标 Accuracy
3. 阈值：默认 0.5 可能不是业务最优——可在验证折扫阈值
4. 失败案例：抽 5 个 FN 看特征是否异常

`FeatureImportanceAnalyzer` 用于解释，不能替代协议。


In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np
# 阈值扫描示意
try:
    model = list(results.values())[0].get('model') if 'results' in globals() else None
except Exception:
    model=None
print('若有 predict_proba，可在验证折上扫阈值最大化 F1。本课重点是协议与代价，而非调到极限。')
print('wanglaoshi API: DataAnalyzer, QuickBaseline, FeatureImportanceAnalyzer, SubmissionGenerator')


## 附录 D：特征组语义

sklearn 乳腺癌数据列名含 `mean` / `error` / `worst`。Worst 往往更可分，但也与 mean 共线。PCA 解释方差曲线帮助决定保留维数。


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
try:
    Xs = StandardScaler().fit_transform(Xtr)
    pca = PCA().fit(Xs)
    cum = np.cumsum(pca.explained_variance_ratio_)
    print('dims for 90% var:', int(np.searchsorted(cum, 0.90)+1))
    print('dims for 95% var:', int(np.searchsorted(cum, 0.95)+1))
except Exception as e:
    print(e)


## 深度阅读与作业（Breast Cancer）

### 必读观点
1. 医学二分类必须谈 **代价非对称**：漏诊（FN）与误报（FP）成本不同。
2. 多重共线特征组（mean/se/worst）使线性系数不稳；树模型 importance 会被相关特征分摊。
3. 阈值 0.5 很少是业务最优；在验证折上约束 Precision 下限再最大化 Recall。

### 书面作业
1. 给出混淆矩阵并计算敏感性/特异度，用一句话写给「非技术科室」。
2. 解释为何先 PCA 再分类可能帮助或伤害可解释性。
3. 使用 `FeatureImportanceAnalyzer` 的结果时，如何避免因果过度解读？

### 参考
sklearn breast cancer dataset docs；通用医疗 ML 检查清单（验证协议、校准、亚组表现）。


## 课堂讨论记录区（请学生填写）

主题：04_breast_cancer

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：04_breast_cancer

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：04_breast_cancer

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：04_breast_cancer

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。
